In [ ]:
# =========================================================
# 2-Stage Demo: Deblur (PyTorch) + Colorize (Keras) in Streamlit
# =========================================================
# pip install streamlit pillow scikit-image opencv-python torch torchvision tensorflow

import os, numpy as np, streamlit as st
from PIL import Image

# ---------------- USER PATHS (edit as needed) ----------------
DEBLUR_WEIGHTS = "/kaggle/working/deblurganv2_brightness_best.pth"  # stage-1 .pth
DEBLUR_FPN_CH = 128
DEBLUR_DEC_CH = 128

COLOR_MODEL_PATH = "colorization_model.h5"  # stage-2 .h5/.keras
COLOR_IMG_SIZE   = 128                      # match your training size

# ---------------- Common helpers ----------------
def pil_to_uint8_rgb(img: Image.Image) -> np.ndarray:
    return np.array(img.convert("RGB"))

def uint8_to_pil(arr: np.ndarray) -> Image.Image:
    return Image.fromarray(arr.astype(np.uint8))

def is_grayscale(rgb_u8: np.ndarray, eps: float = 2.0) -> bool:
    r, g, b = rgb_u8[...,0].astype(np.float32), rgb_u8[...,1].astype(np.float32), rgb_u8[...,2].astype(np.float32)
    diff = np.maximum.reduce([np.abs(r-g), np.abs(r-b), np.abs(g-b)])
    return float(diff.mean()) <= eps

# =========================================================
#                   STAGE 1: DEBLUR (PyTorch)
# =========================================================
import torch, torch.nn as nn
from skimage import color

device_torch = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ResidualBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.c1 = nn.Conv2d(ch, ch, 3, padding=1)
        self.c2 = nn.Conv2d(ch, ch, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(ch)
        self.bn2 = nn.BatchNorm2d(ch)
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        y = self.act(self.bn1(self.c1(x)))
        y = self.bn2(self.c2(y))
        return self.act(x + y)

class SimpleFPN(nn.Module):
    def __init__(self, in_ch=1, fpn_ch=128, dec_ch=128):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(in_ch, fpn_ch, 3, padding=1), nn.ReLU(True))
        self.enc2 = nn.Sequential(nn.Conv2d(fpn_ch, fpn_ch, 3, stride=2, padding=1), nn.ReLU(True))
        self.enc3 = nn.Sequential(nn.Conv2d(fpn_ch, fpn_ch, 3, stride=2, padding=1), nn.ReLU(True))
        self.res  = nn.Sequential(ResidualBlock(fpn_ch), ResidualBlock(fpn_ch))
        self.up2  = nn.ConvTranspose2d(fpn_ch, dec_ch, 2, stride=2)
        self.up1  = nn.ConvTranspose2d(dec_ch, dec_ch, 2, stride=2)
        self.out  = nn.Conv2d(dec_ch, 1, 3, padding=1)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        b  = self.res(e3)
        d2 = self.up2(b)
        d1 = self.up1(d2)
        y  = self.out(d1)
        return torch.sigmoid(y)

class DeblurGANv2Gen(nn.Module):
    def __init__(self, fpn_ch=128, dec_ch=128):
        super().__init__()
        self.net = SimpleFPN(in_ch=1, fpn_ch=fpn_ch, dec_ch=dec_ch)
    def forward(self, x):
        return self.net(x)

# Load generator
G = DeblurGANv2Gen(fpn_ch=DEBLUR_FPN_CH, dec_ch=DEBLUR_DEC_CH).to(device_torch)
if not os.path.exists(DEBLUR_WEIGHTS):
    raise FileNotFoundError(f"Deblur weights not found: {DEBLUR_WEIGHTS}")
G.load_state_dict(torch.load(DEBLUR_WEIGHTS, map_location=device_torch))
G.eval()

def rgb_to_L01(rgb_uint8: np.ndarray) -> torch.Tensor:
    arr = rgb_uint8.astype(np.float32) / 255.0
    Lab = color.rgb2lab(arr).astype(np.float32)
    L01 = np.clip(Lab[...,0] / 100.0, 0, 1)
    return torch.from_numpy(L01).unsqueeze(0).unsqueeze(0).float()

def compose_L_with_original_ab(L_restored01: torch.Tensor, rgb_uint8: np.ndarray) -> np.ndarray:
    L = (L_restored01.squeeze().cpu().numpy() * 100.0).clip(0,100).astype(np.float32)
    arr = rgb_uint8.astype(np.float32) / 255.0
    Lab_in = color.rgb2lab(arr).astype(np.float32)
    Lab_out = np.dstack([L, Lab_in[...,1], Lab_in[...,2]])
    rgb = (color.lab2rgb(Lab_out) * 255.0).clip(0,255).astype(np.uint8)
    return rgb

def run_deblur_pipeline(input_img: Image.Image) -> Image.Image:
    rgb_u8 = pil_to_uint8_rgb(input_img)
    H, W = rgb_u8.shape[:2]
    MAX_SIDE = 1024
    if max(H, W) > MAX_SIDE:
        scale = MAX_SIDE / max(H, W)
        rgb_u8 = np.array(Image.fromarray(rgb_u8).resize((int(W*scale), int(H*scale)), Image.BICUBIC))
    L01 = rgb_to_L01(rgb_u8).to(device_torch)
    with torch.no_grad(), torch.amp.autocast(device_type="cuda", enabled=(device_torch.type=="cuda")):
        L_restored = G(L01)
    rgb_restored = compose_L_with_original_ab(L_restored, rgb_u8)
    return uint8_to_pil(rgb_restored)

# =========================================================
#                STAGE 2: COLORIZE (Keras)
# =========================================================
import tensorflow as tf
from tensorflow.keras.models import load_model

if not os.path.exists(COLOR_MODEL_PATH):
    raise FileNotFoundError(f"Colorization model not found: {COLOR_MODEL_PATH}")
color_net = load_model(COLOR_MODEL_PATH, compile=False)

def preprocess_gray_for_color(rgb_uint8: np.ndarray, size=COLOR_IMG_SIZE):
    im = Image.fromarray(rgb_uint8).resize((size, size), Image.BICUBIC)
    rgb = np.asarray(im).astype(np.float32) / 255.0
    gray = np.dot(rgb[..., :3], [0.2989, 0.5870, 0.1140])[..., None]
    return gray[None, ...]

def run_colorize_pipeline(input_img: Image.Image):
    rgb_u8 = pil_to_uint8_rgb(input_img)
    gray_b = preprocess_gray_for_color(rgb_u8, COLOR_IMG_SIZE)
    pred = color_net.predict(gray_b, verbose=0)[0]
    pred_u8 = (pred * 255.0).clip(0,255).astype(np.uint8)
    return uint8_to_pil(pred_u8)

# =========================================================
#                END-TO-END: Stage1 → Stage2
# =========================================================
def run_end_to_end(input_img: Image.Image):
    if input_img is None:
        return None, "Please upload an image."
    deblurred = run_deblur_pipeline(input_img)
    deb_u8 = pil_to_uint8_rgb(deblurred)
    if is_grayscale(deb_u8, eps=2.0):
        colored = run_colorize_pipeline(deblurred)
        return colored, "Detected grayscale → colorized after deblurring."
    else:
        return deblurred, "Detected color → returned deblurred image."

# =========================================================
#                      STREAMLIT UI
# =========================================================
st.set_page_config(page_title="Two-Stage: Deblur + Colorize", layout="wide")

# Intro
st.title("✨ Two-Stage Image Restoration & Colorization")
st.markdown("""
Welcome to the **Two-Stage Image Pipeline** demo!  

**Pipeline Overview:**  
1. **Stage 1 — Deblur:** Restores blurry images using a PyTorch model.  
2. **Stage 2 — Colorize:** Adds color to grayscale images using a Keras U-Net.  
3. **End-to-End:** Automatically runs Stage 1 → Stage 2, colorizing only if the result is grayscale.  

Upload your image below and choose a stage to see the magic!
""")

# Stage selection
stage = st.radio("Select Stage:", ["Stage 1 — Deblur", "Stage 2 — Colorize", "End-to-End"])

uploaded_file = st.file_uploader("Upload an image", type=["png","jpg","jpeg"])

if uploaded_file is not None:
    input_img = Image.open(uploaded_file).convert("RGB")
    
    # Side-by-side input/output panels
    col1, col2 = st.columns(2)
    with col1:
        st.image(input_img, caption="Input Image", use_column_width=True)

    if stage == "Stage 1 — Deblur":
        if st.button("Run Deblur"):
            result = run_deblur_pipeline(input_img)
            with col2:
                st.image(result, caption="Deblurred Image", use_column_width=True)

    elif stage == "Stage 2 — Colorize":
        if st.button("Run Colorization"):
            result = run_colorize_pipeline(input_img)
            with col2:
                st.image(result, caption="Colorized Image", use_column_width=True)

    elif stage == "End-to-End":
        if st.button("Run Full Pipeline"):
            result, msg = run_end_to_end(input_img)
            with col2:
                st.image(result, caption="Final Output", use_column_width=True)
            st.info(msg)
